# Feature Engineering
Builds and selects user, tweet, and graph features for the bot detection model. Split into three sections:

Section 1 — Core Account + Tweet Behavior Features

- account age
- follower/friend ratio
- profile completeness
- tweet count
- retweet ratio
- hashtag/mention/URL averages
- text length averages
- save base feature dataset

Section 2 — Graph/Network Features
- load follower/friend/edge files
- standardize edges
- compute in-degree/out-degree
- compute graph degree
- compute reciprocity
- optional PageRank/community features
- merge with Round 1 dataset
- save graph-enhanced dataset

Section 3 — AI/Text Authenticity Features
- analyze tweet text for AI-like or synthetic-writing patterns
- compute per-account text authenticity features
- merge with Round 2 dataset
- save final model-ready dataset
- These answer the three questions:

Section 1: What does the account look like?  
Section 2: How is the account connected?  
Section 3: How natural or synthetic does the account’s language look?  


In [1]:
import sys
from pathlib import Path

# Make the project root importable so `utils.*` resolves correctly
_project_root = str(Path.cwd().parent if "notebooks" in str(Path.cwd()) else Path.cwd())
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

import pandas as pd

# Import feature builders
from utils.base_features import (
    ROUND1_FEATURE_COLS,
    ROUND1_META_COLS,
    build_round1_features_for_dataset,
    save_feature_frame,
    print_round1_summary,
    validate_round1,
)

# --- Project root ---
PROJECT_ROOT = Path(_project_root)

# --- Data directories ---
PREPROCESSED_DIR = PROJECT_ROOT / "data" / "pre-processed"
PROCESSED_DIR    = PROJECT_ROOT / "data" / "processed"

# --- Output directories ---
ROUND1_DIR = PROCESSED_DIR / "round1_base"
EDGES_DIR  = PROCESSED_DIR / "edges"           # cached cleaned edge lists
ROUND1_DIR.mkdir(parents=True, exist_ok=True)
EDGES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PREPROCESSED_DIR exists:", PREPROCESSED_DIR.exists())
print("PROCESSED_DIR exists:", PROCESSED_DIR.exists())


PROJECT_ROOT: /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research
PREPROCESSED_DIR exists: True
PROCESSED_DIR exists: True


## Section 1: Base Features
This section builds per-user base features using:
- Account metadata (profile-level features)
- Tweet behavior aggregates (activity + text-derived features)

Output:
- Per-dataset feature files
- Combined feature table across all datasets

In [2]:
DATASETS = {
    "cresci_2015": {
        "users": PREPROCESSED_DIR / "cresci_2015" / "users_cresci_2015.csv",
        "tweets": PREPROCESSED_DIR / "cresci_2015" / "tweets_cresci_2015.csv",
        "reference_date": "2015-12-31",
    },
    "cresci_2017": {
        "users": PREPROCESSED_DIR / "cresci_2017" / "users_cresci_2017.csv",
        "tweets": PREPROCESSED_DIR / "cresci_2017" / "tweets_cresci_2017.csv",
        "reference_date": "2017-12-31",
    },
    "twibot_2020": {
        "users": PREPROCESSED_DIR / "twibot_2020" / "users_twibot_20.csv",
        "tweets": PREPROCESSED_DIR / "twibot_2020" / "tweets_twibot_20.csv",
        "reference_date": "2020-12-31",
    },
    "twibot_2022": {
        "users": PREPROCESSED_DIR / "twibot_2022" / "users_twibot_22.csv",
        "tweets": PREPROCESSED_DIR / "twibot_2022" / "tweets_twibot_22.csv",
        "reference_date": "2022-12-31",
    },
}

for name, paths in DATASETS.items():
    print(name)
    print("  users:", paths["users"].exists(), paths["users"])
    print("  tweets:", paths["tweets"].exists(), paths["tweets"])

cresci_2015
  users: True /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/pre-processed/cresci_2015/users_cresci_2015.csv
  tweets: True /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/pre-processed/cresci_2015/tweets_cresci_2015.csv
cresci_2017
  users: True /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/pre-processed/cresci_2017/users_cresci_2017.csv
  tweets: True /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/pre-processed/cresci_2017/tweets_cresci_2017.csv
twibot_2020
  users: True /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/pre-processed/twibot_2020/users_twibot_20.csv
  tweets: True /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/pre-processed/twibot_2020/tweets_twibot_20.csv
twibot_2022
  users: True /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/pre-processed/twibot_2022/users_twibot_22.csv
  tweets: True /Users

### Build Round 1 Features

In [3]:
round1_frames = {}

for dataset_name, paths in DATASETS.items():
    print(f"\n=== Building Round 1 for {dataset_name} ===")

    df = build_round1_features_for_dataset(dataset_name, paths)

    print("Shape:", df.shape)

    # Parquet preserves dtypes (NaN vs 0 distinction matters here) and is
    # 5-10x faster to reload. Each call also writes a tiny summary so the
    # canonical schema is observable in the notebook output.
    out_path = ROUND1_DIR / f"{dataset_name}_base_features.parquet"
    save_feature_frame(df, out_path)

    print("Saved to:", out_path)
    print(f"  has_tweet_data=1 rate: {df['has_tweet_data'].astype('Int8').mean():.3f}")
    print(f"  retweet_ratio NaN rate: {df['retweet_ratio'].isna().mean():.3f}")
    print(f"  reply_ratio   NaN rate: {df['reply_ratio'].isna().mean():.3f}")

    round1_frames[dataset_name] = df



=== Building Round 1 for cresci_2015 ===


/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/utils/base_features.py:190: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)


Shape: (5301, 36)
Saved to: /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/processed/round1_base/cresci_2015_base_features.parquet
  has_tweet_data=1 rate: 0.971
  retweet_ratio NaN rate: 0.029
  reply_ratio   NaN rate: 0.029

=== Building Round 1 for cresci_2017 ===


/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/utils/base_features.py:190: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)


Shape: (14368, 36)
Saved to: /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/processed/round1_base/cresci_2017_base_features.parquet
  has_tweet_data=1 rate: 0.710
  retweet_ratio NaN rate: 0.290
  reply_ratio   NaN rate: 0.290

=== Building Round 1 for twibot_2020 ===


/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/utils/base_features.py:190: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)


Shape: (11826, 36)
Saved to: /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/processed/round1_base/twibot_2020_base_features.parquet
  has_tweet_data=1 rate: 0.993
  retweet_ratio NaN rate: 0.007
  reply_ratio   NaN rate: 1.000

=== Building Round 1 for twibot_2022 ===
Shape: (1000000, 36)
Saved to: /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/processed/round1_base/twibot_2022_base_features.parquet
  has_tweet_data=1 rate: 0.934
  retweet_ratio NaN rate: 0.066
  reply_ratio   NaN rate: 0.066


### Combine All Datasets

In [4]:
round1_all = pd.concat(round1_frames.values(), ignore_index=True)

print("Combined shape:", round1_all.shape)

out_path = ROUND1_DIR / "all_datasets_base_features.parquet"
save_feature_frame(round1_all, out_path)

print("Saved combined file to:", out_path)

# Hard schema validation — fails loudly if the cross-dataset contract is broken.
diagnostics = validate_round1(round1_all)
for ds, d in diagnostics.items():
    print(f"\n{ds}: rows={d['rows']:,}")
    if d["zero_variance_features"]:
        print(f"  ⚠️ zero-variance features: {d['zero_variance_features']}")
    high_nan = d["nan_rate"][d["nan_rate"] > 0]
    if len(high_nan):
        print("  NaN rate by feature:")
        print(high_nan.to_string())


Combined shape: (1031495, 36)
Saved combined file to: /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/processed/round1_base/all_datasets_base_features.parquet

cresci_2015: rows=5,301
  NaN rate by feature:
has_default_pic      1.0000
verified             1.0000
tweet_active_days    1.0000
avg_text_length      0.0289
retweet_ratio        0.0289
avg_hashtags         0.0289
avg_mentions         0.0289
avg_urls             0.0289
reply_ratio          0.0289

cresci_2017: rows=14,368
  NaN rate by feature:
has_default_pic      1.0000
verified             1.0000
tweet_active_days    1.0000
avg_text_length      0.2903
retweet_ratio        0.2903
avg_hashtags         0.2903
avg_mentions         0.2903
avg_urls             0.2903
reply_ratio          0.2903

twibot_2020: rows=11,826
  NaN rate by feature:
tweet_active_days    1.0000
avg_text_length      0.0068
retweet_ratio        0.0068
avg_hashtags         0.0068
avg_mentions         0.0068
avg_urls             0.0068


### Summary

In [5]:
print_round1_summary(round1_all)

print("\nReply ratio stats:")
print(round1_all["reply_ratio"].describe())

print("\nSample rows with replies:")
display(round1_all[round1_all["reply_ratio"] > 0].head(10))

Rows by dataset:
dataset
twibot_2022    1000000
cresci_2017      14368
twibot_2020      11826
cresci_2015       5301
Name: count, dtype: int64

Label distribution:
label           bot   human      All
dataset                             
cresci_2015    3987    1314     5301
cresci_2017   10894    3474    14368
twibot_2020    6589    5237    11826
twibot_2022  139943  860057  1000000
All          161413  870082  1031495

Round 1 feature columns:
- followers_count
- friends_count
- statuses_count
- listed_count
- log1p_followers
- log1p_friends
- log1p_statuses
- log1p_listed
- followers_per_day
- friends_per_day
- statuses_per_day
- listed_per_day
- account_age_days
- ff_ratio
- has_bio
- has_location
- has_default_pic
- verified
- tweet_count_actual
- log1p_tweet_count
- tweets_per_day
- tweet_active_days
- avg_text_length
- retweet_ratio
- avg_hashtags
- avg_mentions
- avg_urls
- reply_ratio
- has_tweet_data

NaN rate per Round 1 feature, by dataset:
dataset             cresci_2015  c

,dataset,user_id,label,subset,source,followers_count,friends_count,statuses_count,listed_count,log1p_followers,...,tweet_active_days,avg_text_length,retweet_ratio,avg_hashtags,avg_mentions,avg_urls,reply_ratio,has_tweet_data,first_tweet_at,last_tweet_at
0,cresci_2015,24503,human,TFP,cresci_2015,5055,1466,4340,256,8.528331,...,NaN,89.649391,0.054376,0.186066,0.915038,0.472954,0.248938,1,NaT,NaT
1,cresci_2015,22903,human,TFP,cresci_2015,132,194,164,4,4.890349,...,NaN,73.347561,0.121951,0.170732,0.768293,0.091463,0.530488,1,NaT,NaT
2,cresci_2015,382393,human,TFP,cresci_2015,1154,832,1070,92,7.051856,...,NaN,101.381665,0.203929,0.561272,0.853134,0.727783,0.115996,1,NaT,NaT
3,cresci_2015,286543,human,TFP,cresci_2015,930,535,6892,28,6.836259,...,NaN,103.889835,0.386310,0.306992,0.652761,0.540541,0.076968,1,NaT,NaT
4,cresci_2015,438023,human,TFP,cresci_2015,173,444,2885,2,5.159055,...,NaN,77.124133,0.015603,0.042996,0.096394,0.936200,0.011096,1,NaT,NaT
5,cresci_2015,586003,human,TFP,cresci_2015,97,234,216,0,4.584967,...,NaN,92.145631,0.373786,0.776699,1.165049,0.451456,0.223301,1,NaT,NaT
6,cresci_2015,628563,human,TFP,cresci_2015,154,314,505,3,5.043425,...,NaN,68.767068,0.020080,0.102410,0.192771,0.905622,0.090361,1,NaT,NaT
7,cresci_2015,769985,human,TFP,cresci_2015,1100,975,1897,39,7.003974,...,NaN,96.248938,0.164544,0.559448,0.567941,0.677282,0.042463,1,NaT,NaT
8,cresci_2015,2007021,human,TFP,cresci_2015,2254,2196,3052,98,7.720905,...,NaN,95.367018,0.144503,0.445359,0.752798,0.421659,0.263331,1,NaT,NaT
9,cresci_2015,2731681,human,TFP,cresci_2015,31,55,313,1,3.465736,...,NaN,77.318182,0.097403,0.272727,0.327922,0.824675,0.068182,1,NaT,NaT


### Sample Validation

In [6]:
pd.set_option("display.max_columns", None)

round1_all.sample(10)

,dataset,user_id,label,subset,source,followers_count,friends_count,statuses_count,listed_count,log1p_followers,log1p_friends,log1p_statuses,log1p_listed,followers_per_day,friends_per_day,statuses_per_day,listed_per_day,account_age_days,ff_ratio,has_bio,has_location,has_default_pic,verified,tweet_count_actual,log1p_tweet_count,tweets_per_day,tweet_active_days,avg_text_length,retweet_ratio,avg_hashtags,avg_mentions,avg_urls,reply_ratio,has_tweet_data,first_tweet_at,last_tweet_at
477498,twibot_2022,u575183276,human,val,twibot_22,1466,330,718,21,7.290975,5.802118,6.577861,3.091042,0.377058,0.084877,0.184671,0.005401,3887,4.429003,1,1,0,0,47.0,3.871201,0.171106,274.682731,214.702128,0.0,0.489362,2.361702,0.765957,0.148936,1,2021-06-01 13:57:54+00:00,2022-03-03 06:21:02+00:00
725465,twibot_2022,u706658779979345920,bot,train,twibot_22,23,175,1,0,3.178054,5.170484,0.693147,0.000000,0.009237,0.070281,0.000402,0.000000,2489,0.130682,0,0,0,0,1.0,0.693147,1.000000,1.000000,119.000000,0.0,1.000000,2.000000,1.000000,0.000000,1,2016-05-01 23:15:56+00:00,2016-05-01 23:15:56+00:00
848171,twibot_2022,u1003112016,human,train,twibot_22,3981,258,7467,27,8.289539,5.556828,8.918383,3.332205,1.084150,0.070261,2.033497,0.007353,3671,15.370656,1,0,0,1,40.0,3.713572,2.331957,17.152975,140.875000,0.0,2.325000,1.025000,0.000000,0.000000,1,2022-02-08 05:18:05+00:00,2022-02-25 08:58:22+00:00
793349,twibot_2022,u1904737123,human,train,twibot_22,1077,1415,8691,1,6.982863,7.255591,9.070158,0.693147,0.318262,0.418144,2.568262,0.000296,3383,0.760593,1,1,0,0,86.0,4.465908,1.650262,52.112951,83.569767,0.0,0.081395,0.779070,0.418605,0.651163,1,2022-01-13 19:07:53+00:00,2022-03-06 21:50:32+00:00
291675,twibot_2022,u1972905512,human,val,twibot_22,4253,4583,5724,90,8.355615,8.430327,8.652598,4.510860,1.265774,1.363988,1.703571,0.026786,3359,0.927792,1,1,0,0,38.0,3.663562,0.025456,1492.794329,113.236842,0.0,0.421053,1.289474,0.315789,0.236842,1,2017-11-05 09:43:07+00:00,2021-12-07 04:46:57+00:00
682524,twibot_2022,u990044713,human,train,twibot_22,351,1376,1700,0,5.863631,7.227662,7.438972,0.000000,0.095432,0.374116,0.462208,0.000000,3677,0.254902,1,1,0,0,9.0,2.302585,0.044441,202.514201,76.333333,0.0,0.333333,0.666667,0.000000,0.000000,1,2014-12-04 05:45:47+00:00,2015-06-24 18:06:14+00:00
37824,twibot_2022,u1454292521593544708,human,test,twibot_22,2,41,3,0,1.098612,3.737670,1.386294,0.000000,0.004684,0.096019,0.007026,0.000000,426,0.047619,1,1,0,0,10.0,2.397895,10.000000,1.000000,23.000000,0.0,0.000000,0.000000,1.000000,0.000000,1,2022-01-30 18:11:16+00:00,2022-01-30 18:25:47+00:00
706243,twibot_2022,u2207185578,human,train,twibot_22,1486,3997,25027,1,7.304516,8.293550,10.127750,0.693147,0.446649,1.201383,7.522393,0.000301,3326,0.371686,0,0,0,0,40.0,3.713572,40.000000,1.000000,114.875000,0.0,0.225000,1.050000,0.100000,0.025000,1,2022-02-22 21:47:24+00:00,2022-02-23 02:59:34+00:00
905931,twibot_2022,u438156528,human,train,twibot_22,151810,230,72660,132,11.930392,5.442418,11.193560,4.890349,37.641954,0.057030,18.016365,0.032730,4032,657.186147,1,1,0,0,184.0,5.220356,5.557595,33.107847,136.717391,0.0,4.141304,0.288043,1.000000,0.000000,1,2022-02-08 01:20:17+00:00,2022-03-13 03:55:35+00:00
644651,twibot_2022,u2910524586,human,train,twibot_22,64,317,396,1,4.174387,5.762051,5.983936,0.693147,0.021644,0.107203,0.133920,0.000338,2956,0.201258,1,0,0,0,40.0,3.713572,0.131800,303.491019,135.125000,0.0,1.300000,1.250000,0.175000,0.025000,1,2020-09-21 22:01:18+00:00,2021-07-22 09:48:22+00:00


## Section 2: Graph / Network Features
Builds per-user graph features from follower/following/interaction edges.
For each dataset:
- cresci_2015: follow edges (friends.csv + followers.csv per subset)
- cresci_2017: reply interaction edges only (no follow graph exists)
- twibot_2020: follow edges (neighbor field in train/test/dev.json)
- twibot_2022: follow edges (edge-003.csv filtered via DuckDB, 170M rows)

ego_density is set to 0 for twibot_2022 due to 1M-node infeasibility.

In [7]:
from utils.graph_features import (
    GRAPH_FEATURE_COLS,
    load_edges_for_dataset,
    compute_graph_features,
    merge_graph_features,
    merge_embeddings,
    compute_node2vec_embeddings,
    save_graph_frame,
    available_graph_feature_cols,
    compute_label_diagnostics,
)

import networkx as nx
import numpy as np

# Output directories
GRAPH_DIR     = PROCESSED_DIR / "round2_graph"
EMBEDDING_DIR = PROCESSED_DIR / "round3_embeddings"
ROUND2_DIR    = PROCESSED_DIR / "round2_full"
FINAL_DIR     = PROCESSED_DIR / "final_features"

GRAPH_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)
ROUND2_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

print("Graph output dir:", GRAPH_DIR)
print("Embedding output dir:", EMBEDDING_DIR)
print("Round 2 checkpoint dir:", ROUND2_DIR)
print("Final dataset dir:", FINAL_DIR)


RAW_PATHS = {
    "cresci_2015": {
        "raw_graph": PROJECT_ROOT / "data" / "raw" / "cresci-2015",
    },
    "cresci_2017": {
        # No raw graph — only interaction edges from tweets
    },
    "twibot_2020": {
        "raw_graph": PROJECT_ROOT / "data" / "raw" / "Twibot20",
    },
    "twibot_2022": {
        "raw_graph": PROJECT_ROOT / "data" / "raw" / "Twibot22" / "edge-003.csv",
    },
}

Graph output dir: /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/processed/round2_graph
Embedding output dir: /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/processed/round3_embeddings
Round 2 checkpoint dir: /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/processed/round2_full
Final dataset dir: /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/processed/final_features


### Sanity check for one dataset

In [8]:
# Debug single dataset before full loop
test_dataset = "cresci_2015"

edges_test = load_edges_for_dataset(
    dataset_name=test_dataset,
    raw_paths=RAW_PATHS.get(test_dataset, {}),
    preprocessed_paths=DATASETS[test_dataset],
)

print(f"{test_dataset} edges shape:", edges_test.shape)
edges_test.head()

cresci_2015 | E13.csv | friends columns: ['source_id', 'target_id']
cresci_2015 | E13.csv | followers columns: ['source_id', 'target_id']
cresci_2015 | TWT.csv | friends columns: ['source_id', 'target_id']
cresci_2015 | TWT.csv | followers columns: ['source_id', 'target_id']
cresci_2015 | INT.csv | friends columns: ['source_id', 'target_id']
cresci_2015 | INT.csv | followers columns: ['source_id', 'target_id']
cresci_2015 | FSF.csv | friends columns: ['source_id', 'target_id']
cresci_2015 | FSF.csv | followers columns: ['source_id', 'target_id']
cresci_2015 | TFP.csv | friends columns: ['source_id', 'target_id']
cresci_2015 | TFP.csv | followers columns: ['source_id', 'target_id']
cresci_2015 edges shape: (4322027, 4)


,dataset,source_id,target_id,relation
0,cresci_2015,3610511,12,friend
1,cresci_2015,3610511,13,friend
2,cresci_2015,3610511,380,friend
3,cresci_2015,3610511,418,friend
4,cresci_2015,3610511,586,friend


In [9]:
print(edges_test["relation"].value_counts())
print("Unique source users:", edges_test["source_id"].nunique())
print("Unique target users:", edges_test["target_id"].nunique())

relation
friend      2408434
follower    1842831
reply         70762
Name: count, dtype: int64
Unique source users: 1157200
Unique target users: 501924


### Tests feature computation test

In [10]:
gf_test = compute_graph_features(
    edges_df=edges_test,
    dataset_name=test_dataset,
    include_pagerank=True,
    include_ego=True,
)

print("Graph features shape:", gf_test.shape)
gf_test.head()

  Graph too large (1,306,129 nodes > 200,000) — returning NaN ego features
Graph features shape: (1306129, 19)


,dataset,user_id,graph_in_degree,graph_out_degree,graph_degree_total,graph_ff_ratio,graph_reciprocity_count,graph_reciprocity_ratio,graph_pagerank,graph_pagerank_sampled,graph_component_size,graph_neighbor_avg_in_degree,graph_neighbor_avg_out_degree,graph_neighbor_avg_degree_total,graph_neighbor_avg_ff_ratio,graph_neighbor_std_degree_total,graph_ego_node_count,graph_ego_edge_count,graph_ego_density
0,cresci_2015,1000000344,0,1,1,0.000000,0.0,0.000000,NaN,0,1306129,35504.000000,847.000000,36351.000000,41.867925,0.000000,NaN,NaN,NaN
1,cresci_2015,1000000442,0,1,1,0.000000,0.0,0.000000,NaN,0,1306129,468924.000000,911.000000,469835.000000,514.171053,0.000000,NaN,NaN,NaN
2,cresci_2015,1000001444,1,1,2,0.500000,1.0,0.500000,NaN,0,1306129,1965.000000,1916.000000,3881.000000,1.025039,0.000000,NaN,NaN,NaN
3,cresci_2015,1000004431,0,2,2,0.000000,0.0,0.000000,NaN,0,1306129,252214.000000,879.000000,253093.000000,278.019489,306519.475936,NaN,NaN,NaN
4,cresci_2015,1000004564,3,6,9,0.428571,3.0,0.428571,0.000001,1,1306129,2081.222222,1651.888889,3733.111111,1.346883,1313.257062,NaN,NaN,NaN


### Build and Cache Edges

In [11]:
# Edges are expensive to rebuild (twibot_2022's edge-003.csv runs through
# DuckDB; twibot_2020 parses three multi-MB JSONs). Cache cleaned edge
# frames as parquet so subsequent notebook runs are fast.
edges_by_dataset = {}

for dataset_name, paths in DATASETS.items():
    cache_path = EDGES_DIR / f"{dataset_name}_edges.parquet"

    if cache_path.exists():
        print(f"\n=== Loading cached edges for {dataset_name} from {cache_path.name} ===")
        edges = pd.read_parquet(cache_path)
        print(f"Edges loaded from cache: {len(edges):,}")
    else:
        print(f"\n=== Building edges for {dataset_name} ===")
        edges = load_edges_for_dataset(
            dataset_name=dataset_name,
            raw_paths=RAW_PATHS.get(dataset_name, {}),
            preprocessed_paths=paths,
        )
        edges.to_parquet(cache_path, index=False)
        print(f"Edges built and cached: {len(edges):,} → {cache_path}")

    if len(edges) == 0:
        print("  WARNING: no edges found")

    edges_by_dataset[dataset_name] = edges



=== Building edges for cresci_2015 ===
cresci_2015 | E13.csv | friends columns: ['source_id', 'target_id']
cresci_2015 | E13.csv | followers columns: ['source_id', 'target_id']
cresci_2015 | TWT.csv | friends columns: ['source_id', 'target_id']
cresci_2015 | TWT.csv | followers columns: ['source_id', 'target_id']
cresci_2015 | INT.csv | friends columns: ['source_id', 'target_id']
cresci_2015 | INT.csv | followers columns: ['source_id', 'target_id']
cresci_2015 | FSF.csv | friends columns: ['source_id', 'target_id']
cresci_2015 | FSF.csv | followers columns: ['source_id', 'target_id']
cresci_2015 | TFP.csv | friends columns: ['source_id', 'target_id']
cresci_2015 | TFP.csv | followers columns: ['source_id', 'target_id']
Edges built and cached: 4,322,027 → /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/processed/edges/cresci_2015_edges.parquet

=== Building edges for cresci_2017 ===
Edges built and cached: 319,066 → /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_

## Compute graph features

In [12]:
graph_frames = {}

for dataset_name, edges in edges_by_dataset.items():
    print(f"\n=== Graph features for {dataset_name} ===")

    if len(edges) == 0:
        print("Skipping (no edges)")
        continue

    include_ego = dataset_name != "twibot_2022"

    gf = compute_graph_features(
        edges_df=edges,
        dataset_name=dataset_name,
        include_pagerank=True,
        include_ego=include_ego,
    )

    print(f"Graph features shape: {gf.shape}")

    out_path = GRAPH_DIR / f"{dataset_name}_graph_features.parquet"
    save_graph_frame(gf, out_path)

    graph_frames[dataset_name] = gf



=== Graph features for cresci_2015 ===
  Graph too large (1,306,129 nodes > 200,000) — returning NaN ego features
Graph features shape: (1306129, 19)

=== Graph features for cresci_2017 ===
  Graph too large (275,443 nodes > 200,000) — returning NaN ego features
Graph features shape: (275443, 19)

=== Graph features for twibot_2020 ===
Graph features shape: (191562, 19)

=== Graph features for twibot_2022 ===
Graph features shape: (5633910, 19)


## Compute Embeddings

In [13]:
# Note on cross-era use: node2vec embeddings are computed independently
# per-dataset on a sampled subgraph (max_edges=500_000), so the embedding
# spaces are NOT aligned across datasets. Treat `emb_*` as auxiliary
# within-dataset features only — exclude them from the cross-era model
# unless you add an alignment step (e.g. shared anchor users).
embedding_frames = {}

for dataset_name, edges in edges_by_dataset.items():
    print(f"\n=== Embeddings for {dataset_name} ===")

    if len(edges) == 0:
        print("Skipping")
        continue

    emb_df = compute_node2vec_embeddings(
        edges_df=edges,
        dataset_name=dataset_name,
        dimensions=64,
        walk_length=10,
        num_walks=10,
        window=5,
        max_edges=500_000,
    )

    out_path = EMBEDDING_DIR / f"{dataset_name}_embeddings.parquet"
    emb_df.to_parquet(out_path, index=False)

    print(f"Saved: {out_path}")

    embedding_frames[dataset_name] = emb_df



=== Embeddings for cresci_2015 ===


/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[cresci_2015] Sampling edges for Node2Vec: 4,322,027 → 500,000
[cresci_2015] Graph for embeddings: 259,378 nodes, 500,000 edges


Generating walks (CPU: 2): 100%|██████████| 5/5 [00:27<00:00,  5.52s/it]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_

Saved: /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/processed/round3_embeddings/cresci_2015_embeddings.parquet

=== Embeddings for cresci_2017 ===
[cresci_2017] Graph for embeddings: 275,443 nodes, 319,066 edges


...

Saved: /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/processed/round3_embeddings/cresci_2017_embeddings.parquet

=== Embeddings for twibot_2020 ===
[twibot_2020] Graph for embeddings: 191,562 nodes, 208,381 edges


Generating walks (CPU: 2): 100%|██████████| 5/5 [00:03<00:00,  1.36it/s]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Saved: /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/processed/round3_embeddings/twibot_2020_embeddings.parquet

=== Embeddings for twibot_2022 ===
[twibot_2022] Sampling edges for Node2Vec: 17,898,634 → 500,000
[twibot_2022] Graph for embeddings: 564,668 nodes, 500,000 edges


Generating walks (CPU: 2): 100%|██████████| 5/5 [00:08<00:00,  1.63s/it]


Saved: /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/processed/round3_embeddings/twibot_2022_embeddings.parquet


### Combine graph + embeddings

In [14]:
all_graph = pd.concat(
    [df for df in graph_frames.values() if len(df) > 0],
    ignore_index=True
)

all_embeddings = pd.concat(
    [df for df in embedding_frames.values() if len(df) > 0],
    ignore_index=True
)

# Deduplicate embeddings
all_embeddings = all_embeddings.drop_duplicates(
    subset=["dataset", "user_id"]
)

print("Combined graph features:", all_graph.shape)
print("Combined embeddings:", all_embeddings.shape)

Combined graph features: (7407044, 19)
Combined embeddings: (1291051, 66)


### Save intermediaries

In [15]:
save_graph_frame(all_graph, GRAPH_DIR / "all_datasets_graph_features.parquet")
all_embeddings.to_parquet(EMBEDDING_DIR / "all_datasets_embeddings.parquet", index=False)

print("Saved graph features + embeddings")


Saved graph features + embeddings


### Load base features

In [16]:
# Reload from parquet (preserves NaN-vs-0 + dtypes that CSV would lose).
round1_path = PROCESSED_DIR / "round1_base" / "all_datasets_base_features.parquet"
round1_all = pd.read_parquet(round1_path)

print("Round1 path:", round1_path)
print("Round1 shape:", round1_all.shape)


Round1 path: /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/processed/round1_base/all_datasets_base_features.parquet
Round1 shape: (1031495, 36)


### Check types

In [17]:
for df in [round1_all, all_graph, all_embeddings]:
    df["user_id"] = df["user_id"].astype(str)
    df["dataset"] = df["dataset"].astype(str)

print("Type alignment complete")

Type alignment complete


In [18]:
# --- Round 2 checkpoint: base features + graph features ---
round2_all = merge_graph_features(round1_all, all_graph)

out_path = ROUND2_DIR / "all_datasets_round2.parquet"
round2_all.to_parquet(out_path, index=False)

print("Round 2 saved to:", out_path)
print("Round 2 shape:", round2_all.shape)

# After the merge, NaN in graph cols means "we don't know" (e.g. user not
# in graph, or pagerank sampled out). Zero would mean "measured zero." The
# merge logic preserves that distinction.
missing_graph = round2_all[GRAPH_FEATURE_COLS].isna().sum()
print("\nMissing (NaN) graph feature values per column:")
print(missing_graph[missing_graph > 0] if missing_graph.any() else "  None")


Round 2 saved to: /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/processed/round2_full/all_datasets_round2.parquet
Round 2 shape: (1031495, 53)

Missing (NaN) graph feature values per column:
graph_pagerank                      681128
graph_pagerank_sampled               81553
graph_neighbor_avg_in_degree         81553
graph_neighbor_avg_out_degree        81553
graph_neighbor_avg_degree_total      81553
graph_neighbor_avg_ff_ratio          81553
graph_neighbor_std_degree_total      81553
graph_ego_node_count               1019689
graph_ego_edge_count               1019689
graph_ego_density                  1019689
dtype: int64


### Merge graph features + embeddings

In [19]:
merged = merge_embeddings(round2_all, all_embeddings)

### Fill embedded columns

In [20]:
emb_cols = [c for c in merged.columns if c.startswith("emb_")]

if emb_cols:
    merged[emb_cols] = merged[emb_cols].fillna(0)
    print(f"Filled {len(emb_cols)} embedding columns")
else:
    print("WARNING: No embedding columns found")

Filled 64 embedding columns


### Save final dataset

In [21]:
final_path = FINAL_DIR / "all_datasets_full_features.parquet"
merged.to_parquet(final_path, index=False)

print("Final dataset saved to:", final_path)
print("Final dataset shape:", merged.shape)


Final dataset saved to: /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/processed/final_features/all_datasets_full_features.parquet
Final dataset shape: (1031495, 117)


### Summary stats

In [22]:
print("\nGraph feature summary:")
print(
    merged.groupby("dataset")[
        ["graph_in_degree", "graph_pagerank"]
    ].agg(["mean", "median", "max"]).round(4)
)

print("\nEmbedding stats:")
print(merged[emb_cols].describe().round(3))


Graph feature summary:
            graph_in_degree                  graph_pagerank               
                       mean median       max           mean median     max
dataset                                                                   
cresci_2015        348.9344   20.0  468924.0         0.0001    0.0  0.0829
cresci_2017          0.0828    0.0     396.0         0.0000    0.0  0.0010
twibot_2020          9.6880   10.0      97.0         0.0000    0.0  0.0002
twibot_2022          9.1711    2.0   10166.0         0.0000    0.0  0.0003

Embedding stats:
             emb_0        emb_1        emb_2        emb_3        emb_4  \
count  1031495.000  1031495.000  1031495.000  1031495.000  1031495.000   
mean         0.005       -0.008        0.001        0.003        0.001   
std          0.043        0.070        0.040        0.038        0.044   
min         -1.736       -2.753       -2.022       -2.594       -4.510   
25%          0.000        0.000        0.000        0.000      

### Isolation Check

In [23]:
# A user is "isolated" if their zero-meaningful degree features are all 0.
# We don't include the NaN-meaningful columns (pagerank, ego_*, neighbor_*)
# here because NaN would be incorrectly treated as "not equal to 0" and
# inflate the isolation count.
ISOLATION_COLS = [
    "graph_in_degree",
    "graph_out_degree",
    "graph_degree_total",
    "graph_reciprocity_count",
]
isolated = (merged[ISOLATION_COLS].fillna(0) == 0).all(axis=1)

print("\nFully isolated users:")
print(merged[isolated].groupby("dataset")["label"].value_counts())



Fully isolated users:
dataset      label
cresci_2017  bot       6084
             human     2212
twibot_2020  human       17
             bot          3
twibot_2022  human    57410
             bot      15827
Name: count, dtype: int64


### Diagnostics

In [24]:
for dataset_name, edges in edges_by_dataset.items():
    print(f"\n=== Diagnostics for {dataset_name} ===")

    labels = merged[
        merged["dataset"] == dataset_name
    ][["user_id", "label"]]

    labeled_ids = set(labels["user_id"])

    edges_filtered = edges[
        edges["source_id"].isin(labeled_ids) &
        edges["target_id"].isin(labeled_ids)
    ]

    if len(edges_filtered) > 0:
        diag = compute_label_diagnostics(edges_filtered, labels)
        print(diag.describe().round(3))


=== Diagnostics for cresci_2015 ===
       following_bot_ratio  follower_bot_ratio  neighbor_bot_ratio
count             1660.000            1518.000            1986.000
mean                 0.798               0.809               0.828
std                  0.318               0.324               0.305
min                  0.000               0.000               0.000
25%                  0.667               0.714               0.764
50%                  1.000               1.000               1.000
75%                  1.000               1.000               1.000
max                  1.000               1.000               1.000

=== Diagnostics for cresci_2017 ===
       following_bot_ratio  follower_bot_ratio  neighbor_bot_ratio
count              874.000             641.000            1359.000
mean                 0.693               0.574               0.677
std                  0.461               0.492               0.465
min                  0.000               0.000         

### Final cross-dataset feature contract validation

This is the gate the cross-era model relies on:
- the same feature columns exist in every dataset
- `(dataset, user_id)` is unique
- we explicitly call out per-dataset NaN rates and zero-variance features so
  schema drift surfaces here, not silently inside a model fit later.


In [25]:
import numpy as np

CANONICAL_FEATURE_COLS = ROUND1_FEATURE_COLS + GRAPH_FEATURE_COLS

# 1) Uniqueness of (dataset, user_id)
dup = merged.duplicated(subset=["dataset", "user_id"]).sum()
assert dup == 0, f"{dup} duplicate (dataset, user_id) keys in final frame"
print(f"OK: (dataset, user_id) is unique across {len(merged):,} rows")

# 2) Same feature columns exist in every dataset
missing_per_dataset = {}
for ds, g in merged.groupby("dataset"):
    miss = [c for c in CANONICAL_FEATURE_COLS if c not in g.columns]
    if miss:
        missing_per_dataset[ds] = miss
assert not missing_per_dataset, (
    f"feature columns missing per-dataset: {missing_per_dataset}"
)
print(f"OK: all {len(CANONICAL_FEATURE_COLS)} canonical features present in every dataset")

# 3) Per-dataset NaN rate and zero-variance features
print("\nPer-dataset feature health:")
rows = []
for ds, g in merged.groupby("dataset"):
    feat = g[CANONICAL_FEATURE_COLS]
    nan_rate = feat.isna().mean()
    zero_var = []
    for col in CANONICAL_FEATURE_COLS:
        s = pd.to_numeric(feat[col], errors="coerce").dropna()
        if len(s) > 1 and s.nunique() <= 1:
            zero_var.append(col)
    rows.append({
        "dataset": ds,
        "rows": len(g),
        "fully_nan_features": int((nan_rate == 1).sum()),
        "high_nan_features (>50%)": int((nan_rate > 0.5).sum()),
        "zero_variance_features": len(zero_var),
        "zero_var_list": ", ".join(zero_var) if zero_var else "—",
    })
print(pd.DataFrame(rows).to_string(index=False))

# 4) Surface fully-NaN columns explicitly — those are dataset-identity bombs
print("\nFeatures fully NaN within at least one dataset:")
fully_nan_table = (
    merged.groupby("dataset")[CANONICAL_FEATURE_COLS]
          .apply(lambda g: g.isna().mean())
)
flagged = fully_nan_table.columns[(fully_nan_table == 1).any(axis=0)].tolist()
if flagged:
    print(fully_nan_table[flagged].T.round(3).to_string())
    print(
        "\nThese features should be excluded from the cross-era model "
        "(they're a dataset-identity proxy for the dataset where they're "
        "fully NaN)."
    )
else:
    print("  None")


OK: (dataset, user_id) is unique across 1,031,495 rows
OK: all 46 canonical features present in every dataset

Per-dataset feature health:
    dataset    rows  fully_nan_features  high_nan_features (>50%)  zero_variance_features          zero_var_list
cresci_2015    5301                   6                         6                       1   graph_component_size
cresci_2017   14368                   6                        13                       1 graph_pagerank_sampled
twibot_2020   11826                   2                         2                       1 graph_pagerank_sampled
twibot_2022 1000000                   3                         4                       0                      —

Features fully NaN within at least one dataset:
dataset               cresci_2015  cresci_2017  twibot_2020  twibot_2022
has_default_pic             1.000         1.00        0.000        0.000
verified                    1.000         1.00        0.000        0.000
tweet_active_days           

### Cross-era feature contract

`select_cross_era_features` filters the canonical feature set down to
columns that survive the cross-era constraint:
- not in the reference-date-sensitive deny list (raw counts, account_age_days,
  tweet_count_actual — their log1p / per-day companions stay in)
- not embedding columns (per-dataset spaces aren't aligned)
- has < 99% NaN in every dataset
- has > 1 distinct value in every dataset

The surviving list is saved to `feature_schema.json` next to the final
parquet so downstream training code reads it as a contract.


In [26]:
import json
from utils.base_features import select_cross_era_features

CANDIDATE_COLS = (
    ROUND1_FEATURE_COLS
    + GRAPH_FEATURE_COLS
    + [c for c in merged.columns if c.startswith("emb_")]
)

cross_era_cols, excluded = select_cross_era_features(
    merged,
    candidate_cols=CANDIDATE_COLS,
)

print(f"Cross-era feature set: {len(cross_era_cols)} of {len(CANDIDATE_COLS)} candidates kept\n")
print("Kept:")
for c in cross_era_cols:
    print(f"  {c}")

print("\nExcluded:")
for col, reason in excluded:
    print(f"  {col:<40} {reason}")

# Persist as a sidecar contract — downstream training code reads this
# instead of reconstructing the column list.
schema = {
    "all_feature_cols": list(CANDIDATE_COLS),
    "cross_era_feature_cols": cross_era_cols,
    "within_dataset_only_cols": [c for c, _ in excluded],
    "exclusion_reasons": {c: r for c, r in excluded},
    "round1_feature_cols": list(ROUND1_FEATURE_COLS),
    "graph_feature_cols": list(GRAPH_FEATURE_COLS),
    "meta_cols": list(ROUND1_META_COLS),
}
schema_path = FINAL_DIR / "feature_schema.json"
with open(schema_path, "w") as f:
    json.dump(schema, f, indent=2)

print(f"\nSchema contract saved to: {schema_path}")


Cross-era feature set: 31 of 110 candidates kept

Kept:
  log1p_followers
  log1p_friends
  log1p_statuses
  log1p_listed
  followers_per_day
  friends_per_day
  statuses_per_day
  listed_per_day
  ff_ratio
  has_bio
  has_location
  log1p_tweet_count
  tweets_per_day
  avg_text_length
  retweet_ratio
  avg_hashtags
  avg_mentions
  avg_urls
  has_tweet_data
  graph_in_degree
  graph_out_degree
  graph_degree_total
  graph_ff_ratio
  graph_reciprocity_count
  graph_reciprocity_ratio
  graph_pagerank
  graph_neighbor_avg_in_degree
  graph_neighbor_avg_out_degree
  graph_neighbor_avg_degree_total
  graph_neighbor_avg_ff_ratio
  graph_neighbor_std_degree_total

Excluded:
  followers_count                          deny_cols (reference-date sensitive)
  friends_count                            deny_cols (reference-date sensitive)
  statuses_count                           deny_cols (reference-date sensitive)
  listed_count                             deny_cols (reference-date sensitive)
  a